# Email Spam Detection Using Machine Learning
### Exploratory Data Analysis, Model Training, and Evaluation
---
**Dataset:** SMS Spam Collection (UCI / Kaggle)

**Models:** Naive Bayes | Support Vector Machine | Neural Network

**Objective:** Build and compare text classification models to detect spam messages using NLP preprocessing and supervised learning.

---

## Table of Contents
1. Import Libraries
2. Load and Inspect Dataset
3. Class Distribution Analysis
4. Message Length Analysis
5. Most Frequent Words per Class
6. Text Preprocessing and TF-IDF Vectorization
7. Model Training and Evaluation
8. Confusion Matrices
9. Model Comparison Chart
10. Live Prediction Test
11. Conclusion

---
## 1. Import Libraries

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)

pd.set_option('display.max_colwidth', 80)
sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120
print('All libraries imported successfully.')

---
## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv('../data/spam.csv', encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'text']
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print('Dataset Shape:', df.shape)
print('Missing Values:', df.isnull().sum().sum())
print()
print('Class Distribution:')
dist = df['label'].value_counts().reset_index()
dist.columns = ['Class', 'Count']
dist['Percentage'] = (dist['Count'] / len(df) * 100).round(2).astype(str) + '%'
print(dist.to_string(index=False))
print()
df.head(10)

---
## 3. Class Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Class Distribution', fontsize=14, fontweight='bold')

counts = df['label'].value_counts()
colors  = ['#43a047', '#e53935']

axes[0].bar(counts.index, counts.values, color=colors, width=0.4, edgecolor='white')
axes[0].set_title('Message Count per Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Ratio')

plt.tight_layout()
os.makedirs('../reports', exist_ok=True)
plt.savefig('../reports/eda_class_distribution.png')
plt.show()

---
## 4. Message Length Analysis

In [ ]:
df['char_count'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

print('Average Message Statistics by Class:')
stats = df.groupby('label')[['char_count', 'word_count']].agg(['mean', 'max', 'min']).round(1)
print(stats.to_string())
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Message Length Distribution by Class', fontsize=13, fontweight='bold')

for label, color in [('ham', '#43a047'), ('spam', '#e53935')]:
    sub = df[df['label'] == label]
    axes[0].hist(sub['char_count'], bins=40, alpha=0.65, label=label.title(), color=color)
    axes[1].hist(sub['word_count'], bins=30, alpha=0.65, label=label.title(), color=color)

axes[0].set_title('Character Count')
axes[0].set_xlabel('Characters')
axes[0].legend()
axes[1].set_title('Word Count')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/eda_length_distribution.png')
plt.show()

---
## 5. Most Frequent Words per Class

In [ ]:
STOP = set(stopwords.words('english'))

def top_words(series, n=15):
    words = ' '.join(series).lower().split()
    words = [w for w in words if w.isalpha() and w not in STOP]
    return Counter(words).most_common(n)

ham_top  = top_words(df[df['label'] == 'ham']['text'])
spam_top = top_words(df[df['label'] == 'spam']['text'])

print('Top 15 Ham Words:')
print(pd.DataFrame(ham_top, columns=['Word', 'Frequency']).to_string(index=False))
print()
print('Top 15 Spam Words:')
print(pd.DataFrame(spam_top, columns=['Word', 'Frequency']).to_string(index=False))
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Most Frequent Words by Class (excluding stopwords)', fontsize=13, fontweight='bold')

for ax, data, color, title in [
    (axes[0], ham_top,  '#43a047', 'Top 15 Ham Words'),
    (axes[1], spam_top, '#e53935', 'Top 15 Spam Words'),
]:
    words, counts = zip(*data)
    ax.barh(words[::-1], counts[::-1], color=color, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../reports/eda_top_words.png')
plt.show()

---
## 6. Text Preprocessing and TF-IDF Vectorization

In [ ]:
from src.preprocess import load_and_preprocess, clean_text

# Show preprocessing effect on sample messages
samples = df.sample(5, random_state=1)[['label', 'text']].copy()
samples['cleaned'] = samples['text'].apply(clean_text)
print('Preprocessing Sample (Original -> Cleaned):')
print(samples[['label', 'text', 'cleaned']].to_string(index=False))
print()

X_train, X_test, y_train, y_test, vectorizer = load_and_preprocess('../data/spam.csv')

summary = pd.DataFrame({
    'Split':    ['Training Set', 'Test Set'],
    'Samples':  [X_train.shape[0], X_test.shape[0]],
    'Features': [X_train.shape[1], X_test.shape[1]],
})
print('Vectorization Summary:')
print(summary.to_string(index=False))

---
## 7. Model Training and Evaluation

In [ ]:
from src.train import train_all

results = train_all('../data/spam.csv')

# Structured results table
rows = []
for name, m in results.items():
    rows.append({
        'Model':     name.replace('_', ' ').title(),
        'Accuracy':  f"{m['accuracy']*100:.2f}%",
        'Precision': f"{m['precision']*100:.2f}%",
        'Recall':    f"{m['recall']*100:.2f}%",
        'F1-Score':  f"{m['f1']*100:.2f}%",
    })

results_df = pd.DataFrame(rows)
print('=' * 65)
print(' MODEL EVALUATION RESULTS')
print('=' * 65)
print(results_df.to_string(index=False))
print('=' * 65)

---
## 8. Confusion Matrices

In [ ]:
from IPython.display import Image, display
import glob

cm_files = sorted(glob.glob('../reports/cm_*.png'))
fig, axes = plt.subplots(1, len(cm_files), figsize=(5 * len(cm_files), 4))
if len(cm_files) == 1: axes = [axes]

for ax, fpath in zip(axes, cm_files):
    img = plt.imread(fpath)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(os.path.basename(fpath).replace('cm_','').replace('.png','').replace('_',' ').title(),
                 fontweight='bold', fontsize=11)

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/all_confusion_matrices.png', bbox_inches='tight')
plt.show()

---
## 9. Model Comparison Chart

In [ ]:
from IPython.display import Image
Image('../reports/model_comparison.png')

---
## 10. Live Prediction Test

In [ ]:
from src.predict import predict

test_messages = [
    ('Spam', 'Congratulations! You have won a FREE iPhone. Click now to claim!'),
    ('Ham',  'Hey, are we still meeting tomorrow at 3pm?'),
    ('Spam', 'URGENT: Your bank account will be suspended. Verify immediately.'),
    ('Ham',  'The project report is attached. Please review by Friday.'),
    ('Spam', 'Win cash prizes worth $1000! Call now to claim your reward!'),
]

rows = []
for true_label, msg in test_messages:
    for model_name in ['naive_bayes', 'svm', 'neural_network']:
        r = predict(msg, model_name=model_name)
        rows.append({
            'True Label':  true_label,
            'Model':       model_name.replace('_', ' ').title(),
            'Prediction':  r['label'],
            'Confidence':  f"{r['confidence']}%",
            'Correct':     'Yes' if r['label'].lower() == true_label.lower() else 'No',
            'Message':     msg[:50] + '...' if len(msg) > 50 else msg,
        })

pred_df = pd.DataFrame(rows)
print('=' * 90)
print(' LIVE PREDICTION RESULTS')
print('=' * 90)
print(pred_df.to_string(index=False))
print('=' * 90)
print()
accuracy_per_model = pred_df.groupby('Model')['Correct'].apply(lambda x: (x == 'Yes').sum() / len(x) * 100).round(1)
print('Accuracy on Test Messages:')
print(accuracy_per_model.to_string())

---
## 11. Conclusion

The following table summarizes the trade-offs between the three classifiers evaluated in this project:

| Model | Strength | Limitation | Recommended Use |
|-------|----------|------------|-----------------|
| Naive Bayes | Fastest training, lightweight | Lower recall on spam | Baseline, resource-limited environments |
| SVM | Highest precision, real probability scores | Slower than Naive Bayes | Production spam filtering |
| Neural Network | Balanced precision and recall | Requires more computation | Complex or evolving datasets |

**Overall:** SVM and Neural Network both achieve approximately 98% accuracy on this dataset. SVM is preferred for production deployment due to its consistent precision and interpretable confidence scores.

**Dataset Limitation:** The dataset is imbalanced (87% Ham vs 13% Spam), which requires careful interpretation of accuracy metrics. F1-Score is the primary metric of interest.